# AWAS Streaming Application
## Tasks 2.1.2 · 2.1.3 · 2.1.4

This notebook implements the core streaming pipeline:

| Task | Description |
|---|---|
| **2.1.2** | Spark Structured Streaming — Kafka ingestion, watermarked stream-stream joins (A→B, B→C), dropped-pair logging |
| **2.1.3** | MongoDB sink — bulk upsert with `$push`, retry logic, idempotency |
| **2.1.4** | Violation detection — instantaneous (per-camera) and average speed (per-segment), daily record merging |

### Architecture

```
[Kafka: camera-events-A] ──┐
[Kafka: camera-events-B] ──┼──► Spark Structured Streaming ──► MongoDB: violations
[Kafka: camera-events-C] ──┘          │
                                      ├── Instantaneous: speed > camera limit
                                      ├── Average A→B:   Haversine dist / travel time > cam2 limit
                                      └── Average B→C:   Haversine dist / travel time > cam3 limit
```

### Key Design Decisions

| Decision | Choice | Reason |
|---|---|---|
| Join key | `car_plate` + time window | batch_id is local per producer, not shared |
| Distance | Haversine from MongoDB lat/long | Dynamic — no hardcoded km values |
| Watermark | 30 minutes | Handles out-of-order CSV timestamps |
| Join window | 120 seconds | Max travel time at 60 km/h over ~1 km |
| Min travel time | 10 seconds | Filters physically impossible GPS pairs |
| Daily merge | `$addToSet` upsert on `{car_plate, date}` | Spec requirement 2.1.4; atomic document-level write; idempotent against Spark batch replays |
| Queries | 3 separate streaming queries | Instantaneous + A→B avg + B→C avg |


## Section 0 — Environment Setup

Initialises the SparkSession with the Kafka connector package required for Structured Streaming.

### Why `spark.jars.packages` instead of `PYSPARK_SUBMIT_ARGS`

The Kafka connector (`spark-sql-kafka-0-10_2.12:3.3.0`) must be available before any stream
is read. Two approaches exist:

| Approach | Reliability in Docker/Jupyter |
|---|---|
| `os.environ['PYSPARK_SUBMIT_ARGS']` | Unreliable — env var may be ignored if JVM already started |
| `.config('spark.jars.packages', ...)` | Reliable — passed directly into SparkSession at build time |

`spark.jars.packages` is set directly in the builder so Spark downloads the connector
at session creation, not at process start. This guarantees the Kafka data source is
available when `read_kafka_stream()` is called in Section 3.

`spark.stop()` is called first to tear down any existing hidden session (e.g. from a
previous kernel run without full restart), ensuring the new config is not silently
ignored by `getOrCreate()` reusing a stale session.

### SparkSession Parameters

| Parameter | Value | Reason |
|---|---|---|
| `master` | `local[*]` | Uses all available CPU cores in the Docker container |
| `spark.driver.memory` | `1500m` | Stays within Docker 4GB memory limit |
| `spark.sql.shuffle.partitions` | `4` | Reduces shuffle overhead for small streaming dataset |
| `spark.jars.packages` | `spark-sql-kafka-0-10_2.12:3.3.0` | Kafka connector — must match Spark 3.3.0 exactly |
| `stopGracefullyOnShutdown` | `true` | Ensures streaming queries flush state on kernel interrupt |

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'
)

from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('AWAS-TrafficMonitoring')
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0')
    .config('spark.driver.memory', '1500m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.streaming.stopGracefullyOnShutdown', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

Spark 3.3.0 ready


The following sets the Spark submit arguments to download the Kafka connector package at runtime. The package version must exactly match Spark 3.3.0.

In [2]:
import os

# Kafka connector for Spark 3.3.0 
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'
)
print('PYSPARK_SUBMIT_ARGS set')

PYSPARK_SUBMIT_ARGS set


In [3]:
import math
import time

import pandas as pd
from pymongo import MongoClient, UpdateOne
from pymongo.errors import PyMongoError

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, unix_timestamp, lit, expr, udf
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType
)
print('Imports OK')

Imports OK


In [4]:
#  Configuration 
HOST_IP = '192.168.0.175'
KAFKA_BROKER = '192.168.0.175:9092'
MONGO_URI = "mongodb://192.168.0.175:27017/"
MONGO_DB     = 'awas_db'

TOPIC_A = 'camera-events-A'
TOPIC_B = 'camera-events-B'
TOPIC_C = 'camera-events-C'

WATERMARK_DURATION  = '30 minutes'   # late-arrival tolerance
JOIN_WINDOW_SECONDS = 120            # max valid travel time between adjacent cameras (s)
MIN_TRAVEL_SECONDS  = 10             # min physically plausible travel time (s) at ~360 km/h over 1km
MAX_OFFSETS         = 10000          # max Kafka offsets per micro-batch (memory guard)
CHECKPOINT_BASE     = '/home/student/checkpoints'

print('Config loaded')
print(f'  Kafka broker : {KAFKA_BROKER}')
print(f'  MongoDB URI  : {MONGO_URI}')

Config loaded
  Kafka broker : 192.168.0.175:9092
  MongoDB URI  : mongodb://192.168.0.175:27017/


In [5]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('AWAS-TrafficMonitoring')
    .config('spark.driver.memory',              '1500m')
    .config('spark.sql.shuffle.partitions',     '4')
    .config('spark.streaming.stopGracefullyOnShutdown', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready')

Spark 3.3.0 ready


## Section 1 — MongoDB Static Data Loading (Run Once)

Loads the three reference datasets into MongoDB before the stream starts.
- `camera.csv` → `cameras` collection
- `vehicle.csv` → `vehicles` collection (156 duplicate plates deduped — ownership transfers)
- `camera_event_historic.csv` → `camera_events_historic` collection

Indexes are created here so the stream sink can rely on them.
Run this section **once**. Re-running is safe — collections are dropped and reloaded.


In [6]:
client = MongoClient(MONGO_URI)
db     = client[MONGO_DB]

#  cameras:
cam_df  = pd.read_csv('../data/camera.csv')
cam_docs = [
    {
        'camera_id' : int(r['camera_id']),
        'location'  : {'type': 'Point', 'coordinates': [float(r['longitude']), float(r['latitude'])]},
        'latitude'  : float(r['latitude']),
        'longitude' : float(r['longitude']),
        'position'  : float(r['position']),
        'speed_limit': int(r['speed_limit'])
    }
    for _, r in cam_df.iterrows()
]
db.cameras.drop()
db.cameras.insert_many(cam_docs)
db.cameras.create_index([('camera_id', 1)], unique=True)
db.cameras.create_index([('location',  '2dsphere')])
print(f'cameras    : {len(cam_docs)} docs loaded')

#  vehicles 
veh_df = pd.read_csv('../data/vehicle.csv')
veh_df = veh_df.drop_duplicates(subset='car_plate', keep='first')   # 156 dupes from ownership transfers
veh_docs = veh_df.to_dict('records')   # CSV typo 'vechicle_type' preserved intentionally
db.vehicles.drop()
db.vehicles.insert_many(veh_docs)
db.vehicles.create_index([('car_plate', 1)], unique=True)
print(f'vehicles   : {len(veh_docs)} docs loaded (after dedup)')

# camera_events_historic 
hist_df   = pd.read_csv('../data/camera_event_historic.csv')
hist_docs = hist_df.to_dict('records')
db.camera_events_historic.drop()
db.camera_events_historic.insert_many(hist_docs)
db.camera_events_historic.create_index([('car_plate',    1)])
db.camera_events_historic.create_index([('camera_id_start', 1), ('camera_id_end', 1)])
db.camera_events_historic.create_index([('violation_id', 1)], unique=True)
print(f'historic   : {len(hist_docs)} docs loaded')

#  violations indexes (collection written by stream) 
db.violations.create_index([('car_plate', 1), ('date', 1)], unique=True)
db.violations.create_index([('date', 1)])
print('violations : indexes created')

client.close()
print('\nMongoDB static setup complete.')

cameras    : 3 docs loaded
vehicles   : 9844 docs loaded (after dedup)
historic   : 50000 docs loaded
violations : indexes created

MongoDB static setup complete.


## Section 2 — Camera Config and Haversine Distance

Camera metadata (speed limits, coordinates) is loaded dynamically from MongoDB — **no hardcoded values**.
Inter-camera distances are computed using the Haversine formula on `latitude` and `longitude`.
If a camera is relocated and its coordinates updated in MongoDB, all subsequent calculations reflect the change automatically.

### Join Window Justification

| Scenario | Travel Time A→B (~1 km) |
|---|---|
| At 189.7 km/h (max observed, Camera C) | ~19 seconds |
| At 56.4 km/h (min observed, Camera C) | ~64 seconds |
| At 30 km/h (heavy traffic) | ~120 seconds |

Join window set to **120 seconds** to capture slow-moving traffic without excessive state retention.
Pairs with travel time < 10 seconds are discarded as physically impossible (would imply >360 km/h over ~1 km).


In [7]:
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Compute great-circle distance between two GPS coordinates using the Haversine formula.

    Args:
        lat1, lon1 (float): Latitude and longitude of point 1 in decimal degrees.
        lat2, lon2 (float): Latitude and longitude of point 2 in decimal degrees.

    Returns:
        float: Distance in kilometres.
    """
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Load camera config from MongoDB — dynamic, not hardcoded
client  = MongoClient(MONGO_URI)
cameras  = {doc['camera_id']: doc for doc in client[MONGO_DB].cameras.find()}
client.close()

speed_limits = {cid: cam['speed_limit'] for cid, cam in cameras.items()}

# Pre-compute segment distances dynamically
dist_ab = haversine_km(
    cameras[1]['latitude'], cameras[1]['longitude'],
    cameras[2]['latitude'], cameras[2]['longitude']
)
dist_bc = haversine_km(
    cameras[2]['latitude'], cameras[2]['longitude'],
    cameras[3]['latitude'], cameras[3]['longitude']
)

print('Speed limits loaded from MongoDB:', speed_limits)
print(f'Distance Camera 1 → 2 (Haversine): {dist_ab:.4f} km')
print(f'Distance Camera 2 → 3 (Haversine): {dist_bc:.4f} km')

Speed limits loaded from MongoDB: {1: 110, 2: 110, 3: 90}
Distance Camera 1 → 2 (Haversine): 0.9967 km
Distance Camera 2 → 3 (Haversine): 1.0015 km


## Section 3 — Event Schema and Kafka Stream Reader

### Schema

The schema matches the producer payload exactly, including the metadata fields added at publish time (`producer_id`, `produced_at`, `batch_size`).

### startingOffsets: earliest

Set to `earliest` so Spark reads all events already published by the producers, not just new ones. During demo, run producers first, then start the streaming queries.

`maxOffsetsPerTrigger` is capped at 10,000 to prevent a single micro-batch from
consuming too much memory when catching up on a large backlog.

In [8]:
# Schema must match the producer payload exactly
event_schema = StructType([
    StructField('event_id',      StringType()),
    StructField('batch_id',      IntegerType()),
    StructField('car_plate',     StringType()),
    StructField('camera_id',     IntegerType()),
    StructField('timestamp',     StringType()),   # original camera event time
    StructField('speed_reading', DoubleType()),
    StructField('producer_id',   StringType()),
    StructField('produced_at',   StringType()),   # Kafka ingestion time (metadata)
    StructField('batch_size',    IntegerType())   # events in this batch (metadata)
])

def read_kafka_stream(topic):
    """
    Read a Kafka topic as a Spark Structured Streaming DataFrame.

    Parses JSON payload using event_schema, casts timestamp to event_time,
    and drops rows with unparseable timestamps to avoid null join keys.

    Args:
        topic (str): Kafka topic name to subscribe to.

    Returns:
        DataFrame: Parsed streaming DataFrame with event_time column added.
    """
    raw = (
        spark.readStream
        .format('kafka')
        .option('kafka.bootstrap.servers', KAFKA_BROKER)
        .option('subscribe',               topic)
        .option('startingOffsets',         'earliest')
        .option('maxOffsetsPerTrigger',    MAX_OFFSETS)
        .option('failOnDataLoss',          'false')   
        .load()
    )
    return (
        raw
        .selectExpr('CAST(value AS STRING) AS json_value')
        .select(from_json(col('json_value'), event_schema).alias('d'))
        .select('d.*')
        .withColumn('event_time', to_timestamp(col('timestamp')))
        .filter(col('event_time').isNotNull())
    )

print('Stream reader function defined')

Stream reader function defined


## Section 4 — T2.1.4 Instantaneous Violation Detection

A violation is detected when `speed_reading > speed_limit` for the camera that recorded it.
Speed limits are loaded dynamically from MongoDB in Section 2 and inlined as integer literals
in native Spark column expressions — no UDF or broadcast variable is used. This avoids
serialization issues in Structured Streaming where broadcast variable closures can silently
fail in async executor contexts.

All three streams (A, B, C) are unioned before writing so a single `foreachBatch` function
handles all instantaneous violations uniformly across all cameras.


In [9]:
# Read all 3 streams for instantaneous detection
inst_a = read_kafka_stream(TOPIC_A)
inst_b = read_kafka_stream(TOPIC_B)
inst_c = read_kafka_stream(TOPIC_C)

def instant_violations(stream):
    """
    Filter stream for instantaneous speed violations.
    Uses native Spark column expressions — no UDF, no broadcast variable,
    no serialization risk. Speed limits are inlined as integer literals
    sourced from the speed_limits dict loaded from MongoDB.

    Args:
        stream (DataFrame): Parsed Kafka streaming DataFrame with event_time.

    Returns:
        DataFrame: Filtered stream containing only instantaneous violation rows.
    """
    return (
        stream
        .filter(
            ((col('camera_id') == 1) & (col('speed_reading') > speed_limits[1])) |
            ((col('camera_id') == 2) & (col('speed_reading') > speed_limits[2])) |
            ((col('camera_id') == 3) & (col('speed_reading') > speed_limits[3]))
        )
        .select(
            col('car_plate'),
            col('camera_id').alias('camera_id_start'),
            col('camera_id').alias('camera_id_end'),
            col('event_time').alias('timestamp_start'),
            col('event_time').alias('timestamp_end'),
            col('speed_reading'),
            lit('INSTANTANEOUS').alias('violation_type')
        )
    )

# Union all three streams — one query handles all cameras
violations_instant_stream = (
    instant_violations(inst_a)
    .union(instant_violations(inst_b))
    .union(instant_violations(inst_c))
)

print('Instantaneous violation stream defined')

Instantaneous violation stream defined


### Why No UDF Is Used

Speed limits are written directly into the Spark filter expressions rather than using Python UDFs or broadcast variables. In Structured Streaming, UDFs and broadcast closures can fail in asynchronous executor contexts. Native Spark expressions are more reliable and faster because execution remains within the JVM.


## Section 5 — T2.1.2 Stream-Stream Joins (A→B and B→C)

### Join Strategy

Two separate watermarked stream-stream inner joins are used — one per camera segment.

| Join | Segment | Ending Camera Limit |
|---|---|---|
| A → B | Camera 1 → Camera 2 | 110 km/h |
| B → C | Camera 2 → Camera 3 | 90 km/h |

### Watermark and State Management

A 30-minute watermark is applied to both sides of each join. Spark uses this to:
1. Handle out-of-order event arrivals (confirmed by EDA: timestamps not ordered within batches)
2. Evict state for unmatched events after 30 minutes — prevents unbounded memory growth
3. Guarantee that matched pairs with `event_time` differences > 30 minutes are not produced

Events that do not find a matching partner within the join window are silently evicted from state when the watermark advances past them. These are logged per batch in the sink function.

### Dropped Pair Handling

Within each `foreachBatch` call, pairs with physically impossible travel times (< 10 seconds) are counted and logged before writing to MongoDB. Non-violating valid pairs are also counted for transparency.


In [10]:
#  A → B join
join_a = read_kafka_stream(TOPIC_A).withWatermark('event_time', WATERMARK_DURATION)
join_b_ab = read_kafka_stream(TOPIC_B).withWatermark('event_time', WATERMARK_DURATION)

joined_ab = (
    join_a.alias('a')
    .join(
        join_b_ab.alias('b'),
        expr(f"""
            a.car_plate = b.car_plate
            AND b.event_time > a.event_time
            AND b.event_time <= a.event_time + interval {JOIN_WINDOW_SECONDS} seconds
        """),
        'inner'
    )
    .select(
        col('a.car_plate').alias('car_plate'),
        col('a.camera_id').alias('camera_id_start'),
        col('b.camera_id').alias('camera_id_end'),
        col('a.event_time').alias('timestamp_start'),
        col('b.event_time').alias('timestamp_end'),
        col('a.speed_reading').alias('speed_start'),
        col('b.speed_reading').alias('speed_end')
    )
    .withColumn(
        'travel_time_hours',
        (unix_timestamp('timestamp_end') - unix_timestamp('timestamp_start')) / lit(3600.0)
    )
    .withColumn('speed_reading', lit(dist_ab) / col('travel_time_hours'))
    .withColumn('violation_type', lit('AVERAGE'))
)

print('A→B joined stream defined')

# B → C join:
join_b_bc = read_kafka_stream(TOPIC_B).withWatermark('event_time', WATERMARK_DURATION)
join_c    = read_kafka_stream(TOPIC_C).withWatermark('event_time', WATERMARK_DURATION)

joined_bc = (
    join_b_bc.alias('b')
    .join(
        join_c.alias('c'),
        expr(f"""
            b.car_plate = c.car_plate
            AND c.event_time > b.event_time
            AND c.event_time <= b.event_time + interval {JOIN_WINDOW_SECONDS} seconds
        """),
        'inner'
    )
    .select(
        col('b.car_plate').alias('car_plate'),
        col('b.camera_id').alias('camera_id_start'),
        col('c.camera_id').alias('camera_id_end'),
        col('b.event_time').alias('timestamp_start'),
        col('c.event_time').alias('timestamp_end'),
        col('b.speed_reading').alias('speed_start'),
        col('c.speed_reading').alias('speed_end')
    )
    .withColumn(
        'travel_time_hours',
        (unix_timestamp('timestamp_end') - unix_timestamp('timestamp_start')) / lit(3600.0)
    )
    .withColumn('speed_reading', lit(dist_bc) / col('travel_time_hours'))
    .withColumn('violation_type', lit('AVERAGE'))
)

print('B→C joined stream defined')

A→B joined stream defined
B→C joined stream defined


**Note on B→C vs A→B output volume:**

A→B typically produces more matched pairs per micro-batch than B→C. Camera A and B events share a closer time range and are produced at the same cadence, so many pairs land in the same processing window.

In contrast, Camera C events arrive later in the dataset timeline and often fall into different micro-batches compared to their B counterparts, reducing per-batch join output. This is expected behavior. The 30-minute watermark ensures late C arrivals are still matched as long as all three producers run concurrently.

## Section 6 — T2.1.3 MongoDB Sink

### Design

| Feature | Implementation |
|---|---|
| Daily merge | `UpdateOne` upsert on `{car_plate, date}` — one document per vehicle per day |
| Append violations | `$addToSet` into `violations` array — atomic at document level, deduplicates identical embedded documents |
| Bulk writes | `bulk_write(ordered=False)` — all rows in one network round trip |
| Retry logic | Exponential backoff: 1s → 2s → 4s, max 3 attempts |
| Idempotency | `$addToSet` guarantees that Spark batch replays cannot produce duplicate violation entries. Since `timestamp_start` is part of each embedded doc, genuine separate violations at different times are still appended; only true byte-identical duplicates are blocked. |
| Connection per batch | `MongoClient` created and closed inside `foreachBatch` — not serialised across workers |
| Batch processing | Single `.collect()` per batch with in-memory Python classification — minimises Spark actions and shuffle overhead |

### Dropped Pair Logging

Each average speed batch logs:
- Total joined pairs received
- Pairs dropped for impossible travel time (< 10s)
- Valid non-violating pairs
- Violations written to MongoDB


In [11]:
def make_avg_sink(distance_km, end_camera_id, label):
    """
    Factory that returns a foreachBatch function for average speed violations.

    Collects the batch once, classifies pairs in Python (invalid/non-violation/violation),
    logs the breakdown for transparency, and bulk-upserts violations into MongoDB.

    Args:
        distance_km (float): Haversine distance between the two cameras in km.
        end_camera_id (int): Camera ID at the segment end (used for speed limit lookup).
        label (str): Segment label for logging (e.g. 'A→B').

    Returns:
        function: foreachBatch-compatible sink.
    """
    end_limit = speed_limits[end_camera_id]
    min_hours = MIN_TRAVEL_SECONDS / 3600.0

    def write_batch(batch_df, batch_id):
        # Single Spark action — collect once, classify in Python afterwards
        all_rows = batch_df.collect()
        if not all_rows:
            print(f'[{label} | Batch {batch_id}] Empty batch.')
            return

        # Classify in-memory — zero further Spark actions
        invalid_rows  = [r for r in all_rows if r.travel_time_hours <  min_hours]
        valid_rows = [r for r in all_rows if r.travel_time_hours >= min_hours]
        v_rows  = [r for r in valid_rows if r.speed_reading >  end_limit]
        non_viol_rows = [r for r in valid_rows if r.speed_reading <= end_limit]

        print(f'[{label} | Batch {batch_id}] '
              f'invalid(dropped)={len(invalid_rows)} | '
              f'valid non-violations={len(non_viol_rows)} | '
              f'violations={len(v_rows)}')

        if not v_rows:
            return

        client = MongoClient(MONGO_URI)
        col_ref = client[MONGO_DB].violations
        ops = []

        for row in v_rows:
            vdate  = row.timestamp_start.strftime('%Y-%m-%d')
            detail = {
                'violation_type' : row.violation_type,
                'camera_id_start': int(row.camera_id_start),
                'camera_id_end' : int(row.camera_id_end),
                'timestamp_start': row.timestamp_start,
                'timestamp_end' : row.timestamp_end,
                'speed_reading' : round(float(row.speed_reading), 2)
            }
            ops.append(UpdateOne(
                {'car_plate': row.car_plate, 'date': vdate},
                {
                    '$setOnInsert': {'car_plate': row.car_plate, 'date': vdate},
                    '$addToSet' : {'violations': detail}   # idempotent against replays
                },
                upsert=True
            ))

        for attempt in range(3):
            try:
                res = col_ref.bulk_write(ops, ordered=False)
                print(f'[{label} | Batch {batch_id}] Written: '
                      f'upserted={res.upserted_count} modified={res.modified_count}')
                break
            except PyMongoError as e:
                wait = 2 ** attempt
                if attempt < 2:
                    print(f'[{label} | Retry {attempt+1}] {e} — retrying in {wait}s')
                    time.sleep(wait)
                else:
                    print(f'[{label} | FAILED] {e}')

        client.close()

    return write_batch

def write_instant_batch(batch_df, batch_id):
    """
    foreachBatch sink for instantaneous speed violations.

    Collects the batch, builds a bulk upsert operation per violation,
    and writes to MongoDB using $addToSet for idempotency. Retries
    up to 3 times with exponential backoff on write failure.

    Args:
        batch_df (DataFrame): Spark DataFrame for this micro-batch.
        batch_id (int): Micro-batch sequence number assigned by Spark.
    """
    rows = batch_df.collect()
    if not rows:
        print(f'[INSTANT | Batch {batch_id}] Empty batch.')
        return

    client  = MongoClient(MONGO_URI)
    col_ref = client[MONGO_DB].violations
    ops   = []

    for row in rows:
        vdate  = row.timestamp_start.strftime('%Y-%m-%d')
        detail = {
            'violation_type' : 'INSTANTANEOUS',
            'camera_id_start': int(row.camera_id_start),
            'camera_id_end'  : int(row.camera_id_end),
            'timestamp_start': row.timestamp_start,
            'timestamp_end'  : row.timestamp_end,
            'speed_reading'  : round(float(row.speed_reading), 2)
        }
        ops.append(UpdateOne(
            {'car_plate': row.car_plate, 'date': vdate},
            {
                '$setOnInsert': {'car_plate': row.car_plate, 'date': vdate},
                '$addToSet'   : {'violations': detail}   # idempotent against replayy
            },
            upsert=True
        ))

    for attempt in range(3):
        try:
            res = col_ref.bulk_write(ops, ordered=False)
            print(f'[INSTANT | Batch {batch_id}] Written: '
                  f'{len(ops)} violations | '
                  f'upserted={res.upserted_count} modified={res.modified_count}')
            break
        except PyMongoError as e:
            wait = 2 ** attempt
            if attempt < 2:
                print(f'[INSTANT | Retry {attempt+1}] {e} — retrying in {wait}s')
                time.sleep(wait)
            else:
                print(f'[INSTANT | FAILED] {e}')

    client.close()

print('Sink functions defined')


Sink functions defined


## Section 7 -Start Streaming Queries

Three queries are started:

1. **`q_instant`** -instantaneous violations from all three camera streams
2. **`q_ab`** -average speed violations on segment Camera 1 → Camera 2
3. **`q_bc`** -average speed violations on segment Camera 2 → Camera 3

Each query writes to the same `violations` collection via its respective sink.
All queries run concurrently. `awaitAnyTermination()` blocks until any query stops or fails.

**For demo:** interrupt the kernel after observing output from all three queries. Check MongoDB after running:
```python
client = MongoClient(MONGO_URI)
print(client['awas_db'].violations.count_documents({}))
```

**Checkpoint directories** are used to maintain state across restarts. Delete `checkpoints/` to start fresh.


In [ ]:
# Query 1: Instantaneous violations 
q_instant = (
    violations_instant_stream
    .writeStream
    .outputMode('append')
    .foreachBatch(write_instant_batch)
    .option('checkpointLocation', f'{CHECKPOINT_BASE}/instant')
    .trigger(processingTime='10 seconds')
    .start()
)
print('q_instant started')

#  Query 2: A→B average speed violations 
q_ab = (
    joined_ab
    .writeStream
    .outputMode('append')
    .foreachBatch(make_avg_sink(dist_ab, 2, 'A→B'))
    .option('checkpointLocation', f'{CHECKPOINT_BASE}/avg_ab')
    .trigger(processingTime='10 seconds')
    .start()
)
print('q_ab started')

# Query 3: B→C average speed violations 
q_bc = (
    joined_bc
    .writeStream
    .outputMode('append')
    .foreachBatch(make_avg_sink(dist_bc, 3, 'B→C'))
    .option('checkpointLocation', f'{CHECKPOINT_BASE}/avg_bc')
    .trigger(processingTime='10 seconds')
    .start()
)
print('q_bc started')

print('\nAll 3 queries running. Interrupt kernel to stop.')
print('Monitoring... (use Spark UI at http://localhost:4040 to inspect)')

try:
    spark.streams.awaitAnyTermination()
except KeyboardInterrupt:
    print('\nInterrupted by user.')
finally:
    for q in [q_instant, q_ab, q_bc]:
        q.stop()
    print('All queries stopped.')

q_instant started
q_ab started
q_bc started

All 3 queries running. Interrupt kernel to stop.
Monitoring... (use Spark UI at http://localhost:4040 to inspect)
[INSTANT | Batch 259] Written: 4851 violations | upserted=0 modified=0
[INSTANT | Batch 260] Written: 14 violations | upserted=0 modified=0
[INSTANT | Batch 261] Written: 33 violations | upserted=0 modified=0
[A→B | Batch 254] invalid(dropped)=0 | valid non-violations=3203 | violations=3323
[B→C | Batch 254] invalid(dropped)=0 | valid non-violations=13 | violations=39
[B→C | Batch 254] Written: upserted=0 modified=0
[A→B | Batch 254] Written: upserted=0 modified=0
[INSTANT | Batch 262] Written: 11 violations | upserted=0 modified=0
[B→C | Batch 255] Empty batch.
[A→B | Batch 255] invalid(dropped)=0 | valid non-violations=56 | violations=56
[A→B | Batch 255] Written: upserted=0 modified=0
[INSTANT | Batch 263] Written: 2 violations | upserted=0 modified=0
[B→C | Batch 256] Empty batch.
[A→B | Batch 256] invalid(dropped)=0 | valid 

[B→C | Batch 288] Empty batch.
[INSTANT | Batch 295] Written: 61 violations | upserted=0 modified=0
[A→B | Batch 288] invalid(dropped)=0 | valid non-violations=9 | violations=72
[A→B | Batch 288] Written: upserted=0 modified=0
[B→C | Batch 289] Empty batch.
[INSTANT | Batch 296] Written: 41 violations | upserted=0 modified=0
[A→B | Batch 289] invalid(dropped)=0 | valid non-violations=36 | violations=18
[A→B | Batch 289] Written: upserted=0 modified=0
[B→C | Batch 290] Empty batch.
[INSTANT | Batch 297] Written: 48 violations | upserted=0 modified=0
[A→B | Batch 290] invalid(dropped)=0 | valid non-violations=36 | violations=54
[A→B | Batch 290] Written: upserted=0 modified=0
[B→C | Batch 291] Empty batch.
[INSTANT | Batch 298] Written: 57 violations | upserted=0 modified=0
[A→B | Batch 291] invalid(dropped)=0 | valid non-violations=54 | violations=18
[A→B | Batch 291] Written: upserted=0 modified=0
[B→C | Batch 292] Empty batch.
[INSTANT | Batch 299] Written: 38 violations | upserted=0 

[INSTANT | Batch 331] Written: 71 violations | upserted=0 modified=0
[B→C | Batch 325] Empty batch.
[A→B | Batch 324] invalid(dropped)=0 | valid non-violations=45 | violations=0
[INSTANT | Batch 332] Written: 49 violations | upserted=0 modified=0
[A→B | Batch 325] invalid(dropped)=0 | valid non-violations=0 | violations=36
[A→B | Batch 325] Written: upserted=0 modified=0
[B→C | Batch 326] Empty batch.
[INSTANT | Batch 333] Written: 52 violations | upserted=0 modified=0
[B→C | Batch 327] Empty batch.
[A→B | Batch 326] invalid(dropped)=0 | valid non-violations=54 | violations=27
[A→B | Batch 326] Written: upserted=0 modified=0
[INSTANT | Batch 334] Written: 60 violations | upserted=0 modified=0
[A→B | Batch 327] invalid(dropped)=0 | valid non-violations=27 | violations=45
[B→C | Batch 328] Empty batch.
[A→B | Batch 327] Written: upserted=0 modified=0
[INSTANT | Batch 335] Written: 44 violations | upserted=0 modified=0
[A→B | Batch 328] invalid(dropped)=0 | valid non-violations=36 | viola

[A→B | Batch 353] invalid(dropped)=0 | valid non-violations=18 | violations=72
[A→B | Batch 353] Written: upserted=0 modified=0
[B→C | Batch 354] invalid(dropped)=0 | valid non-violations=4 | violations=8
[B→C | Batch 354] Written: upserted=0 modified=4
[INSTANT | Batch 361] Written: 50 violations | upserted=1 modified=4
[B→C | Batch 355] invalid(dropped)=0 | valid non-violations=2 | violations=10
[B→C | Batch 355] Written: upserted=0 modified=5
[A→B | Batch 354] invalid(dropped)=0 | valid non-violations=36 | violations=45
[A→B | Batch 354] Written: upserted=0 modified=0
[INSTANT | Batch 362] Written: 49 violations | upserted=1 modified=7
[B→C | Batch 356] invalid(dropped)=0 | valid non-violations=4 | violations=20
[B→C | Batch 356] Written: upserted=0 modified=10
[A→B | Batch 355] invalid(dropped)=0 | valid non-violations=63 | violations=36
[A→B | Batch 355] Written: upserted=0 modified=0
[INSTANT | Batch 363] Written: 58 violations | upserted=0 modified=4
[B→C | Batch 357] invalid(dr

[A→B | Batch 378] invalid(dropped)=0 | valid non-violations=48 | violations=32
[A→B | Batch 378] Written: upserted=0 modified=0
[INSTANT | Batch 386] Written: 60 violations | upserted=0 modified=3
[B→C | Batch 380] invalid(dropped)=0 | valid non-violations=6 | violations=4
[B→C | Batch 380] Written: upserted=0 modified=2
[INSTANT | Batch 387] Written: 47 violations | upserted=1 modified=3
[A→B | Batch 379] invalid(dropped)=0 | valid non-violations=16 | violations=40
[A→B | Batch 379] Written: upserted=0 modified=0
[B→C | Batch 381] invalid(dropped)=0 | valid non-violations=0 | violations=10
[B→C | Batch 381] Written: upserted=1 modified=4
[INSTANT | Batch 388] Written: 46 violations | upserted=0 modified=4
[A→B | Batch 380] invalid(dropped)=0 | valid non-violations=24 | violations=32
[A→B | Batch 380] Written: upserted=0 modified=0
[B→C | Batch 382] invalid(dropped)=0 | valid non-violations=4 | violations=8
[B→C | Batch 382] Written: upserted=1 modified=3
[INSTANT | Batch 389] Written:

[INSTANT | Batch 414] Written: 49 violations | upserted=42 modified=7
[A→B | Batch 404] invalid(dropped)=0 | valid non-violations=40 | violations=64
[A→B | Batch 404] Written: upserted=1 modified=7
[INSTANT | Batch 415] Written: 45 violations | upserted=34 modified=11
[B→C | Batch 404] invalid(dropped)=0 | valid non-violations=4 | violations=16
[B→C | Batch 404] Written: upserted=0 modified=8
[A→B | Batch 405] invalid(dropped)=0 | valid non-violations=48 | violations=80
[INSTANT | Batch 416] Written: 56 violations | upserted=48 modified=8
[A→B | Batch 405] Written: upserted=0 modified=10
[B→C | Batch 405] invalid(dropped)=0 | valid non-violations=8 | violations=4
[B→C | Batch 405] Written: upserted=0 modified=2
[INSTANT | Batch 417] Written: 37 violations | upserted=35 modified=2
[A→B | Batch 406] invalid(dropped)=0 | valid non-violations=48 | violations=8
[A→B | Batch 406] Written: upserted=0 modified=1
[INSTANT | Batch 418] Written: 67 violations | upserted=56 modified=11
[B→C | Batc

[INSTANT | Batch 450] Written: 54 violations | upserted=41 modified=13
[B→C | Batch 426] invalid(dropped)=0 | valid non-violations=2 | violations=18
[B→C | Batch 426] Written: upserted=0 modified=9
[INSTANT | Batch 451] Written: 38 violations | upserted=30 modified=8
[A→B | Batch 427] invalid(dropped)=0 | valid non-violations=48 | violations=80
[A→B | Batch 427] Written: upserted=0 modified=10
[INSTANT | Batch 452] Written: 59 violations | upserted=53 modified=6
[B→C | Batch 427] invalid(dropped)=0 | valid non-violations=6 | violations=14
[B→C | Batch 427] Written: upserted=0 modified=7
[INSTANT | Batch 453] Written: 36 violations | upserted=31 modified=5
[A→B | Batch 428] invalid(dropped)=0 | valid non-violations=96 | violations=56
[A→B | Batch 428] Written: upserted=0 modified=7
[INSTANT | Batch 454] Written: 67 violations | upserted=50 modified=17
[B→C | Batch 428] invalid(dropped)=0 | valid non-violations=8 | violations=18
[B→C | Batch 428] Written: upserted=0 modified=9
[A→B | Bat

[A→B | Batch 450] invalid(dropped)=0 | valid non-violations=40 | violations=72
[A→B | Batch 450] Written: upserted=0 modified=9
[INSTANT | Batch 485] Written: 39 violations | upserted=36 modified=3
[B→C | Batch 448] invalid(dropped)=0 | valid non-violations=4 | violations=10
[B→C | Batch 448] Written: upserted=0 modified=5
[A→B | Batch 451] invalid(dropped)=0 | valid non-violations=24 | violations=80
[A→B | Batch 451] Written: upserted=1 modified=9
[INSTANT | Batch 486] Written: 53 violations | upserted=42 modified=11
[B→C | Batch 449] invalid(dropped)=0 | valid non-violations=2 | violations=16
[B→C | Batch 449] Written: upserted=0 modified=8
[INSTANT | Batch 487] Written: 51 violations | upserted=48 modified=3
[A→B | Batch 452] invalid(dropped)=0 | valid non-violations=72 | violations=32
[A→B | Batch 452] Written: upserted=2 modified=2
[B→C | Batch 450] invalid(dropped)=0 | valid non-violations=6 | violations=8
[B→C | Batch 450] Written: upserted=1 modified=3
[INSTANT | Batch 488] Wri

[B→C | Batch 470] invalid(dropped)=0 | valid non-violations=6 | violations=10
[B→C | Batch 470] Written: upserted=0 modified=5
[A→B | Batch 474] invalid(dropped)=0 | valid non-violations=40 | violations=32
[A→B | Batch 474] Written: upserted=0 modified=4
[INSTANT | Batch 519] Written: 60 violations | upserted=51 modified=9
[B→C | Batch 471] invalid(dropped)=0 | valid non-violations=8 | violations=10
[B→C | Batch 471] Written: upserted=0 modified=5
[INSTANT | Batch 520] Written: 68 violations | upserted=57 modified=11
[A→B | Batch 475] invalid(dropped)=0 | valid non-violations=40 | violations=24
[A→B | Batch 475] Written: upserted=0 modified=3
[INSTANT | Batch 521] Written: 33 violations | upserted=26 modified=7
[B→C | Batch 472] invalid(dropped)=0 | valid non-violations=4 | violations=18
[B→C | Batch 472] Written: upserted=0 modified=9
[INSTANT | Batch 522] Written: 67 violations | upserted=59 modified=8
[A→B | Batch 476] invalid(dropped)=0 | valid non-violations=88 | violations=40
[A→

[INSTANT | Batch 556] Written: 87 violations | upserted=74 modified=13
[B→C | Batch 492] invalid(dropped)=0 | valid non-violations=6 | violations=20
[B→C | Batch 492] Written: upserted=0 modified=10
[A→B | Batch 496] invalid(dropped)=0 | valid non-violations=84 | violations=77
[A→B | Batch 496] Written: upserted=0 modified=11
[INSTANT | Batch 557] Written: 117 violations | upserted=104 modified=13
[B→C | Batch 493] invalid(dropped)=0 | valid non-violations=10 | violations=28
[B→C | Batch 493] Written: upserted=0 modified=14
[INSTANT | Batch 558] Written: 108 violations | upserted=97 modified=11
[A→B | Batch 497] invalid(dropped)=0 | valid non-violations=77 | violations=63
[A→B | Batch 497] Written: upserted=1 modified=8
[INSTANT | Batch 559] Written: 74 violations | upserted=62 modified=12
[B→C | Batch 494] invalid(dropped)=0 | valid non-violations=8 | violations=20
[B→C | Batch 494] Written: upserted=0 modified=10
[INSTANT | Batch 560] Written: 76 violations | upserted=65 modified=11


In [ ]:
import shutil, os

CHECKPOINT_BASE = '/home/student/checkpoints'

for folder in ["instant", "avg_ab", "avg_bc"]:
    path = f"{CHECKPOINT_BASE}/{folder}"
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Cleared: {path}")
    else:
        print(f"Not found: {path}")

## Verification - Check MongoDB After Running

Run this cell after stopping the queries to confirm violations were written correctly.


In [ ]:
# Quick verification -check violation counts in MongoDB
client = MongoClient(MONGO_URI)
_db     = client[MONGO_DB]

total_docs  = _db.violations.count_documents({})
inst_count = _db.violations.count_documents({'violations.violation_type': 'INSTANTANEOUS'})
avg_count  = _db.violations.count_documents({'violations.violation_type': 'AVERAGE'})

print(f'Total daily violation documents : {total_docs}')
print(f'Docs with INSTANTANEOUS violations: {inst_count}')
print(f'Docs with AVERAGE violations      : {avg_count}')

# Sample document
sample = _db.violations.find_one()
if sample:
    import pprint
    print('\nSample document:')
    pprint.pprint(sample)

client.close()